In [1]:
import pandas as pd
import numpy as np

# 1. Carregar o banco que você limpou
df = pd.read_csv('SteamGames_CleanByMe.csv') # Ajuste o nome para o seu arquivo final

# 2. Separar as tags por vírgula e "explodir" a coluna para linhas individuais
df['Tag_Split'] = df['Tags'].astype(str).str.split(',')
df_tags = df.explode('Tag_Split')
df_tags['Tag_Split'] = df_tags['Tag_Split'].str.strip()

# 3. Remover registros onde a tag ficou como nula ou vazia
df_tags = df_tags[df_tags['Tag_Split'] != 'nan']

In [2]:
# Agrupar por Tag e calcular métricas cruciais
analise_tendencias = df_tags.groupby('Tag_Split').agg(
    quantidade_jogos=('Appid', 'count'),
    mediana_reviews_positivas=('PositiveReview', 'median'),
    media_taxa_aprovacao=('ApprovalRate', 'mean'),
    preco_medio=('price_clean', 'mean')
).reset_index()

# Filtrar para evitar nichos fantasmas (pegar tags que aparecem em pelo menos 30 jogos)
# Ordenar pelas marcas com maior engajamento de público (Mediana de reviews positivos)
top_oportunidades = analise_tendencias[analise_tendencias['quantidade_jogos'] >= 30]\
    .sort_values(by='mediana_reviews_positivas', ascending=False)

print("Top 15 Características/Tags Promissoras no Mercado Atual:")
print(top_oportunidades.head(15))

Top 15 Características/Tags Promissoras no Mercado Atual:
                     Tag_Split  quantidade_jogos  mediana_reviews_positivas  \
138                       Epic                53                      460.0   
168             Games Workshop                66                      337.0   
273  Open World Survival Craft               282                      331.5   
208                Kickstarter                91                      319.0   
175           Great Soundtrack              1873                      318.0   
245                  Motocross                30                      271.5   
144         Extraction Shooter                57                      270.0   
103               Cult Classic               146                      249.5   
246                  Motorbike                77                      244.0   
316                     Remake               190                      242.0   
253                      Musou                33                      236

In [3]:
# Filtrar jogos recentes que custam menos de 20 dólares e ordenar por sucesso absoluto
jogos_referencia = df[
    (df['ReleaseYear'] >= 2024) & 
    (df['price_clean'] <= 20.0)
].sort_values(by='PositiveReview', ascending=False)

print("\nPrincipais referências de mercado de baixo custo de produção (2024-2026):")
print(jogos_referencia[['Name', 'ReleaseYear', 'price_clean', 'PositiveReview', 'ReviewScore']].head(10))


Principais referências de mercado de baixo custo de produção (2024-2026):
                                  Name  ReleaseYear  price_clean  \
30                          Schedule I       2025.0        19.99   
218            Hollow Knight: Silksong       2025.0        19.99   
40                            R.E.P.O.       2025.0         9.99   
34                                PEAK       2025.0         4.95   
177                            Balatro       2024.0        14.99   
9     Warhammer 40,000: Space Marine 2       2024.0        19.79   
260                           Megabonk       2025.0         9.99   
1326                        WEBFISHING       2024.0         4.99   
491                      My Summer Car       2025.0        14.99   
174                 The Planet Crafter       2024.0        14.39   

      PositiveReview  ReviewScore  
30            176699            9  
218           126480            8  
40            116773            9  
34            112743            

As tags que combinam alta taxa de aprovação, bom engajamento e um preço médio convidativo para o mercado indie destacam fortemente:

Cult Classic (146 jogos, mediana de 249.5 reviews positivos, 85.8% de aprovação, preço médio de $12.94).
Villain Protagonist (Protagonista Vilão) (138 jogos, mediana de 228.0 reviews positivos, 82.6% de aprovação, preço médio de $12.77).

Com esses dados da para concluir que jogos focados em nichos específicos, com narrativas ousadas, mecânicas estilizadas (estilo Carrion, Cult of the Lamb, ou simuladores de gerenciamento sombrios) têm uma comunidade extremamente fiel. Jogos com forte visão autoral.

Sucessos de bilheteria recentes que mantiveram o preço abaixo de $20, tirando os gigantes consolidados que distorcem um pouco a lista (como Silksong e Space Marine 2):

Balatro ($14.99): Um jogo feito basicamente com cartas e uma lógica matemática com elementos de Roguelike (pôquer). Baixíssimo custo de produção de assets, mas um design de loop perfeito.

R.E.P.O. ($9.99) e PEAK ($4.95): Jogos que surfam na onda cooperativa, de exploração tensa ou horror psicológico em primeira pessoa. Geralmente usam estéticas minimalistas, low-poly ou retrô (estilo PS1) que reduzem o tempo de modelagem 3D a quase zero.

In [4]:
# 1. Criar listas a partir das strings separadas por vírgula
df['Tag_list'] = df['Tags'].astype(str).str.split(',').apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else [])
df['Genre_list'] = df['Genres'].astype(str).str.split(',').apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else [])

# 2. Filtrar apenas os jogos que se encaixam no nosso alvo de "Baixo Custo / Alto Engajamento"
df_alvo = df[df['Tag_list'].apply(lambda tags: 'Cult Classic' in tags or 'Villain Protagonist' in tags)].copy()

# 3. Explodir os gêneros para analisar cada categoria separadamente
df_generos_cruzados = df_alvo.explode('Genre_list')

# 4. Agrupar e gerar os insights estatísticos
stats_generos_alvo = df_generos_cruzados.groupby('Genre_list').agg(
    quantidade_jogos=('Appid', 'count'),
    mediana_reviews_positivas=('PositiveReview', 'median'),
    taxa_aprovacao_media=('ApprovalRate', 'mean'),
    preco_medio=('price_clean', 'mean')
).reset_index().sort_values(by='quantidade_jogos', ascending=False)

print("Gêneros ideais para envelopar as temáticas em alta:")
print(stats_generos_alvo.head(7))

Gêneros ideais para envelopar as temáticas em alta:
    Genre_list  quantidade_jogos  mediana_reviews_positivas  \
6        Indie               136                      209.0   
1    Adventure               128                      333.5   
0       Action               124                      255.0   
10  Simulation                78                      138.5   
2       Casual                71                       95.0   
12    Strategy                70                      248.0   
8          RPG                50                      294.5   

    taxa_aprovacao_media  preco_medio  
6               0.841889    10.153971  
1               0.833476    13.430156  
0               0.862197    13.310565  
10              0.809576    10.917308  
2               0.852111     9.231831  
12              0.835294    12.703714  
8               0.826072    14.651400  


O mapa um estúdio solo. A direção mais segura em termos de mercado e tempo de desenvolvimento aponta para:

Mecânica Macro (Genre): Adventure / Strategy (Aventura Narrativa ou Estratégia Minimalista em turnos).

Temática / Estética (Tags): Villain Protagonist ou Cult Classic. Um design visual estilizado (Pixel Art ou Low-Poly/Retrô) focado em uma atmosfera única, onde as escolhas do jogador ou a perspectiva de controlar o "lado sombrio" geram engajamento instantâneo.

Preço Alvo: Entre $9.99 e $14.99. É a zona de conforto onde o público indie compra se o conceito for bem amarrado.

Muito interessante e dessa vez evitei usar graficos para ver se apenas os dados podem apontar algo e realmente podem, ter uma imagem geral e cavar fundo quanto algumas metricas e padrões que fogem o esperado pode dar resultados bem interessantes, principalmente em um banco de dados que foi devidamente limpo para um fim exploratorio para tomada de desição que ocasionalmente apresenta uma direção solida e firme para um estudio solo por exemplo, definindo uma projeção solida que pode guiar tomadas de decisões eficientes e produtivas.